Python 中以双下划线开头和结尾的方法（__method__）被称为特殊方法或魔术方法。它们在特定操作或内置函数调用时被自动触发，用于定制对象的行为。下面按功能分类介绍最常用的方法，并附带演示示例。

## 一、对象生命周期
| 方法 |    触发时机    | 作用 |
|:--------:|:----------:|:--------:|
| __new__(cls, ...) | 实例创建前（类方法） | 控制实例的生成，常用于不可变类型子类或单例模式 |
| __init__(self, ...) |   实例创建后    | 初始化实例属性 |
| __del__(self) |  对象被垃圾回收前  | 释放资源（不推荐依赖，执行时机不确定） |

In [2]:
class LifecycleDemo:
    def __new__(cls, *args, **kwargs):
        print("1. __new__ 创建实例")
        instance = super().__new__(cls)
        return instance

    def __init__(self, name, age):
        print("2. __init__ 初始化实例")
        self.name = name
        self.age = age

    def __del__(self):
        print(f"3. __del__ 销毁实例 {self.name}")


demo = LifecycleDemo(name="zhangsan", age=20)
del demo
"""
注意：
当你写 obj = MyClass(42, name='test') 时，参数 42 和 name='test' 会先传递给 __new__，然后 __new__ 返回的实例会自动把这些参数再传给 __init__。如果 __new__ 不接收 *args, **kwargs，你就无法获取用户传入的参数，也无法决定是否要修改它们或根据它们返回不同的实例。
"""

1. __new__ 创建实例
2. __init__ 初始化实例
3. __del__ 销毁实例 zhangsan


### 二、字符串与表示

| 方法 | 触发时机 | 作用 |
|:-------:|:--------:|:-------:|
| __str__(self) | str(obj)、print(obj) | 返回用户友好的字符串 |
| __repr__(self) | repr(obj)、交互式环境直接显示 | 返回开发者友好的字符串（通常可 eval 还原） |
| __format__(self, spec) | format(obj, spec) | 自定义格式化 |
| __bytes__(self) | bytes(obj) | 返回字节串 |

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def __str__(self):
        return f"Person: {self.name}, {self.age}岁"

    def __repr__(self):
        return f"Person('{self.name}', {self.age})"

p = Person("Alice", 30)
print(str(p))   # Person: Alice, 30岁
print(repr(p))  # Person('Alice', 30)

### 三、属性访问控制

方法	&nbsp;&nbsp;&nbsp;触发时机&nbsp;&nbsp;&nbsp;	作用<br>
__getattr__(self, name)	&nbsp;&nbsp;&nbsp;访问不存在的属性时&nbsp;&nbsp;&nbsp;	返回默认值或抛异常<br>
__getattribute__(self, name)	&nbsp;&nbsp;&nbsp;访问任何属性时（优先）&nbsp;&nbsp;&nbsp;	完全控制属性获取，慎用（易无限递归）<br>
__setattr__(self, name, value)	&nbsp;&nbsp;&nbsp;设置属性时	&nbsp;&nbsp;&nbsp;校验、转换或记录赋值操作<br>
__delattr__(self, name)	&nbsp;&nbsp;&nbsp;删除属性时&nbsp;&nbsp;&nbsp;	防止删除或执行额外操作<br>

In [ ]:
class Guard:
    def __init__(self):
        self.public = "公开"

    def __getattr__(self, name):
        return f"属性 {name} 不存在，返回默认值"

    def __setattr__(self, name, value):
        if name == "secret":
            raise AttributeError("不允许设置 secret 属性")
        super().__setattr__(name, value)  # 必须调用父类方法避免递归

    def __delattr__(self, name):
        print(f"正在删除 {name}")
        super().__delattr__(name)

g = Guard()
print(g.public)   # 公开
print(g.unknown)  # 属性 unknown 不存在，返回默认值
g.public = "新值"  # 正常
# g.secret = 123   # 抛出 AttributeError
del g.public       # 正在删除 public

这里解释下为什么必须调用父类方法避免递归，因为当你写 self.name = value 时，Python 实际上会调用 self.__setattr__(name, value)。如果你已经在 __setattr__ 方法内部写了这样的赋值语句（self.name = value），就会导致：
__setattr__ 被调用 → 内部执行 self.attr = val → 再次调用 __setattr__ → 无限循环
##### 如何避免？
必须使用父类（通常是 object）的原生 __setattr__ 方法，它直接操作实例的 __dict__，而不会再次触发当前类的 __setattr__。
* super().__setattr__(name, value) 调用了父类的 __setattr__，绕过了当前类的重写版本。
* 等价写法：object.__setattr__(self, name, value)。

### 四、容器与序列（模拟列表/字典）

方法	&nbsp;&nbsp;&nbsp;触发时机&nbsp;&nbsp;&nbsp;	作用 <br>
__len__(self)&nbsp;&nbsp;&nbsp;	len(obj)&nbsp;&nbsp;&nbsp;	返回容器长度<br>
__getitem__(self, key)&nbsp;&nbsp;&nbsp;	obj[key]	&nbsp;&nbsp;&nbsp;获取元素<br>
__setitem__(self, key, value)&nbsp;&nbsp;&nbsp;	obj[key] = value	&nbsp;&nbsp;&nbsp;设置元素<br>
__delitem__(self, key)&nbsp;&nbsp;&nbsp;	del obj[key]	&nbsp;&nbsp;&nbsp;删除元素<br>
__contains__(self, item)&nbsp;&nbsp;&nbsp;	item in obj	&nbsp;&nbsp;&nbsp;判断成员关系<br>
__iter__(self)&nbsp;&nbsp;&nbsp;	iter(obj) 或 for x in obj&nbsp;&nbsp;&nbsp;	返回迭代器<br>
__next__(self)&nbsp;&nbsp;&nbsp;	next(iterator)&nbsp;&nbsp;&nbsp;	定义迭代器的下一个元素<br>
__reversed__(self)&nbsp;&nbsp;&nbsp;	reversed(obj)&nbsp;&nbsp;&nbsp;	反向迭代<br>

In [ ]:
class MyList:
    def __init__(self, items):
        self._items = list(items)

    def __len__(self):
        return len(self._items)

    def __getitem__(self, index):
        return self._items[index]

    def __setitem__(self, index, value):
        self._items[index] = value

    def __delitem__(self, index):
        del self._items[index]

    def __contains__(self, item):
        return item in self._items

    def __iter__(self):
        return iter(self._items)

ml = MyList([10, 20, 30])
print(len(ml))        # 3
print(ml[1])          # 20
ml[1] = 25
print(20 in ml)       # False
del ml[0]
for x in ml:
    print(x)          # 25, 30

### 五、比较运算符

方法&nbsp;&nbsp;&nbsp;	触发时机&nbsp;&nbsp;&nbsp;	作用<br>
__lt__(self, other)&nbsp;&nbsp;&nbsp;	self < other&nbsp;&nbsp;&nbsp;	小于   <br>
__le__(self, other)&nbsp;&nbsp;&nbsp;	self <= other&nbsp;&nbsp;&nbsp;	小于等于<br>
__eq__(self, other)&nbsp;&nbsp;&nbsp;	self == other&nbsp;&nbsp;&nbsp;	等于<br>
__ne__(self, other)&nbsp;&nbsp;&nbsp;	self != other&nbsp;&nbsp;&nbsp;	不等于<br>
__gt__(self, other)&nbsp;&nbsp;&nbsp;	self > other&nbsp;&nbsp;&nbsp;	大于<br>
__ge__(self, other)&nbsp;&nbsp;&nbsp;	self >= other&nbsp;&nbsp;&nbsp;	大于等于<br>

In [ ]:
class Score:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        return self.value == other.value

    def __lt__(self, other):
        return self.value < other.value

a = Score(85)
b = Score(90)
print(a == b)   # False
print(a < b)    # True

### 七、可调用对象

方法	&nbsp;&nbsp;&nbsp;触发时机&nbsp;&nbsp;&nbsp;	作用<br>
__call__(self, ...)	obj(...)&nbsp;&nbsp;&nbsp; 像函数一样调用&nbsp;&nbsp;&nbsp;	使实例可调用，常用于装饰器或带状态的函数<br>

In [ ]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, x):
        return x * self.factor

double = Multiplier(2)
print(double(5))   # 10